In [ ]:
import numpy as np
import scipy.optimize

a = np.random.randn(100)[:, None]
b = (np.random.randn(10) + 2)[:, None]
ab = np.r_[a, b]

In [ ]:
import numpy as np
import pulp


def optbox(samples, n):
    N, k = samples.shape
    ok, ng = samples[:n], samples[n:]
    M = samples.max() + 1e3
    prob = pulp.LpProblem("MinWidthBox", pulp.LpMinimize)
    l = pulp.LpVariable.dicts("l", range(k), cat="Continuous")
    u = pulp.LpVariable.dicts("u", range(k), cat="Continuous")
    b = pulp.LpVariable.dicts("b", (range(N - n), range(k)), cat="Binary")
    prob += pulp.lpSum(u[i] - l[i] for i in range(k))
    for x in ok:
        for i in range(k):
            prob += l[i] <= x[i]
            prob += x[i] <= u[i]
    for j, x in enumerate(ng):
        prob += pulp.lpSum(b[j][i] for i in range(k)) >= 1
        for i in range(k):
            prob += x[i] - l[i] >= -M * (1 - b[j][i])
            prob += u[i] - x[i] >= -M * b[j][i]
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    return np.array([[pulp.value(_[i]) for i in range(k)] for _ in [l, u]])
